# Iris MLOps Pipeline on Google Cloud Storage

Tasks 3-6: train from GCS data, store timestamped artifacts, run a separate inference script, run twice, compare data versions.

```
gs://BUCKET/data/<version>/{raw,train,eval}.csv
gs://BUCKET/runs/<UTC timestamp>/{model.joblib, metrics.json, train.log, predictions.csv, inference_metrics.json}
```
`train.py` and `infer.py` are written by cells below and run as separate scripts.

## Setup

In [1]:
! pip3 install --quiet google-cloud-storage pandas scikit-learn joblib

In [2]:
import io, json, os, subprocess, sys, time
import pandas as pd
from google.cloud import storage
from sklearn.model_selection import train_test_split

PROJECT_ID = "project-1fcf3ddd-1680-4d4c-aae"
LOCATION = "asia-south1"
DATA_BUCKET = "mlops_w1_21f3003066_data_bucket"   # <-- CHANGE: the bucket you already created (name only, no gs://)
ARTIFACT_BUCKET = "mlops_w1_21f3003066_artifacts"              # set to a different bucket if you want artifacts kept separately

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET = "species"

client = storage.Client(project=PROJECT_ID)
print("Project:", client.project)

# Confirm read/write access to the bucket(s)
for b in {DATA_BUCKET, ARTIFACT_BUCKET}:
    blob = client.bucket(b).blob("healthcheck.txt")
    blob.upload_from_string("ok"); blob.delete()
    print("OK, writable:", b)

Project: project-1fcf3ddd-1680-4d4c-aae
OK, writable: mlops_w1_21f3003066_data_bucket
OK, writable: mlops_w1_21f3003066_artifacts


## Task 2 (data in GCS): split and upload

Stratified 80/20 split so every species appears in both sets. Each data version has its own folder.

In [3]:
# Use data/iris.csv from the course repo if present; otherwise build an equivalent file from scikit-learn.
if not os.path.exists("data/iris.csv"):
    from sklearn.datasets import load_iris
    os.makedirs("data", exist_ok=True)
    iris = load_iris(as_frame=True)
    df = iris.frame
    df.columns = FEATURES + [TARGET]
    df[TARGET] = df[TARGET].map(dict(enumerate(iris.target_names)))
    df.to_csv("data/iris.csv", index=False)
    print("Created data/iris.csv from scikit-learn (replace with the course file if provided)")
else:
    print("Found data/iris.csv")

def prepare_data(csv_path, version, eval_size=0.2):
    df = pd.read_csv(csv_path)
    train, ev = train_test_split(df, test_size=eval_size, stratify=df[TARGET], random_state=42)
    train.to_csv("train.csv", index=False)
    ev.to_csv("eval.csv", index=False)
    b = client.bucket(DATA_BUCKET)
    base = f"data/{version}"
    b.blob(f"{base}/raw.csv").upload_from_filename(csv_path)
    b.blob(f"{base}/train.csv").upload_from_filename("train.csv")
    b.blob(f"{base}/eval.csv").upload_from_filename("eval.csv")
    print(f"[{version}] uploaded train={len(train)}, eval={len(ev)} to gs://{DATA_BUCKET}/{base}/")

prepare_data("data/iris.csv", "v1")

Found data/iris.csv
[v1] uploaded train=120, eval=30 to gs://mlops_w1_21f3003066_data_bucket/data/v1/


## Task 3: Training script

Reads train/eval data from GCS and writes the model, metrics and log to `runs/<timestamp>/`.

In [4]:
%%writefile train.py
"""Training pipeline: reads train/eval data from GCS, writes artifacts to
gs://<artifact-bucket>/runs/<UTC timestamp>/ (model, metrics, log)."""
import argparse, io, json, logging
from datetime import datetime, timezone

import joblib
import pandas as pd
from google.cloud import storage
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET = "species"

p = argparse.ArgumentParser()
p.add_argument("--project", required=True)
p.add_argument("--data-bucket", required=True)
p.add_argument("--artifact-bucket", required=True)
p.add_argument("--data-version", default="v1")
p.add_argument("--max-depth", type=int, default=3)
a = p.parse_args()

client = storage.Client(project=a.project)

def read_csv(bucket, path):
    return pd.read_csv(io.BytesIO(client.bucket(bucket).blob(path).download_as_bytes()))

run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out = f"runs/{run_ts}"

log_buf = io.StringIO()
fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
handlers = [logging.StreamHandler(), logging.StreamHandler(log_buf)]
log = logging.getLogger("train")
log.setLevel(logging.INFO)
for h in handlers:
    h.setFormatter(fmt)
    log.addHandler(h)

base = f"data/{a.data_version}"
log.info("Run %s | reading gs://%s/%s", run_ts, a.data_bucket, base)
train = read_csv(a.data_bucket, f"{base}/train.csv")
ev = read_csv(a.data_bucket, f"{base}/eval.csv")
log.info("train=%d rows, eval=%d rows", len(train), len(ev))

model = DecisionTreeClassifier(max_depth=a.max_depth, random_state=1)
model.fit(train[FEATURES], train[TARGET])
acc = metrics.accuracy_score(ev[TARGET], model.predict(ev[FEATURES]))
log.info("Eval accuracy: %.4f", acc)

joblib.dump(model, "model.joblib")
bkt = client.bucket(a.artifact_bucket)
bkt.blob(f"{out}/model.joblib").upload_from_filename("model.joblib")
bkt.blob(f"{out}/metrics.json").upload_from_string(json.dumps({
    "run_ts": run_ts, "data_version": a.data_version, "max_depth": a.max_depth,
    "n_train": len(train), "n_eval": len(ev), "eval_accuracy": acc}, indent=2))
log.info("Artifacts written to gs://%s/%s/", a.artifact_bucket, out)
bkt.blob(f"{out}/train.log").upload_from_string(log_buf.getvalue())
print(f"RUN_TS={run_ts}")

Writing train.py


## Task 4: Separate inference script

Downloads the trained model from the run folder in GCS and predicts on the evaluation set.

In [5]:
%%writefile infer.py
"""Inference script: downloads the trained model from the GCS artifact bucket
and runs it on the evaluation set. Results are saved in the same run folder."""
import argparse, io, json

import joblib
import pandas as pd
from google.cloud import storage
from sklearn import metrics

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET = "species"

p = argparse.ArgumentParser()
p.add_argument("--project", required=True)
p.add_argument("--data-bucket", required=True)
p.add_argument("--artifact-bucket", required=True)
p.add_argument("--data-version", default="v1")
p.add_argument("--run-ts", default=None, help="Run folder name; latest if omitted")
a = p.parse_args()

client = storage.Client(project=a.project)

run_ts = a.run_ts
if not run_ts:
    it = client.list_blobs(a.artifact_bucket, prefix="runs/", delimiter="/")
    list(it)
    runs = sorted(x[len("runs/"):].strip("/") for x in it.prefixes)
    run_ts = runs[-1]
out = f"runs/{run_ts}"

bkt = client.bucket(a.artifact_bucket)
bkt.blob(f"{out}/model.joblib").download_to_filename("model_downloaded.joblib")
model = joblib.load("model_downloaded.joblib")

ev = pd.read_csv(io.BytesIO(client.bucket(a.data_bucket)
                 .blob(f"data/{a.data_version}/eval.csv").download_as_bytes()))
ev["prediction"] = model.predict(ev[FEATURES])
acc = metrics.accuracy_score(ev[TARGET], ev["prediction"])

bkt.blob(f"{out}/predictions.csv").upload_from_string(ev.to_csv(index=False))
bkt.blob(f"{out}/inference_metrics.json").upload_from_string(json.dumps({
    "run_ts": run_ts, "data_version": a.data_version, "accuracy": acc,
    "report": metrics.classification_report(ev[TARGET], ev["prediction"], output_dict=True)}, indent=2))
print(f"Run {run_ts} | data {a.data_version} | inference accuracy {acc:.4f}")

Writing infer.py


## Task 5: Run training and inference twice (data v1)

In [6]:
COMMON = ["--project", PROJECT_ID, "--data-bucket", DATA_BUCKET, "--artifact-bucket", ARTIFACT_BUCKET]

def run(script, *extra):
    r = subprocess.run([sys.executable, script, *COMMON, *extra], capture_output=True, text=True)
    print(r.stdout, r.stderr[-1500:] if r.returncode else "")
    r.check_returncode()
    return r.stdout

def train_and_infer(version):
    out = run("train.py", "--data-version", version)
    ts = [l for l in out.splitlines() if l.startswith("RUN_TS=")][-1].split("=", 1)[1]
    run("infer.py", "--data-version", version, "--run-ts", ts)
    return ts

runs = []
for _ in range(2):
    runs.append(train_and_infer("v1"))
    time.sleep(2)  # ensures a distinct timestamp folder
print("Runs:", runs)

RUN_TS=20261011T143747Z
 
Run 20261011T143747Z | data v1 | inference accuracy 0.9667
 
RUN_TS=20261011T143800Z
 
Run 20261011T143800Z | data v1 | inference accuracy 0.9667
 
Runs: ['20261011T143747Z', '20261011T143800Z']


## Task 6 (Optional): Multiple data versions

Set the paths to the data files from the GitHub data folder, then run.

In [7]:
DATA_VERSIONS = {
    "v1": "data/iris.csv",
    # "v2": "data/iris_v2.csv",   # <- replace with the actual file names from the GitHub data folder
}

for version, path in DATA_VERSIONS.items():
    if version != "v1":  # v1 already uploaded and run above
        prepare_data(path, version)
        train_and_infer(version)
        time.sleep(2)

In [8]:
rows = []
for blob in client.list_blobs(ARTIFACT_BUCKET, prefix="runs/"):
    if blob.name.endswith("/metrics.json"):
        m = json.loads(blob.download_as_text())
        inf = client.bucket(ARTIFACT_BUCKET).blob(blob.name.replace("metrics.json", "inference_metrics.json"))
        m["inference_accuracy"] = json.loads(inf.download_as_text())["accuracy"] if inf.exists() else None
        rows.append(m)

results = pd.DataFrame(rows).sort_values("run_ts").reset_index(drop=True)
results

,run_ts,data_version,max_depth,n_train,n_eval,eval_accuracy,inference_accuracy
0,20261011T143224Z,v1,3,120,30,0.966667,0.966667
1,20261011T143238Z,v1,3,120,30,0.966667,0.966667
2,20261011T143747Z,v1,3,120,30,0.966667,0.966667
3,20261011T143800Z,v1,3,120,30,0.966667,0.966667


In [9]:
# Verify the timestamped folders in the bucket
! gsutil ls -r gs://{ARTIFACT_BUCKET}/runs/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://mlops_w1_21f3003066_artifacts/runs/:

gs://mlops_w1_21f3003066_artifacts/runs/20261011T143224Z/:
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143224Z/inference_metrics.json
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143224Z/metrics.json
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143224Z/model.joblib
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143224Z/predictions.csv
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143224Z/train.log

gs://mlops_w1_21f3003066_artifacts/runs/20261011T143238Z/:
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143238Z/inference_metrics.json
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143238Z/metrics.json
gs://mlops_w1_21f3003066_artifacts/runs/20261011T143238Z/model.joblib

In [10]:
zip -r runs.zip runs

SyntaxError: invalid syntax (284624112.py, line 1)

## Notes
- Fixed seeds make repeated runs on the same data give identical metrics; the two runs produce two separate timestamped folders.
- The eval set is small (about 30 rows), so one misclassified sample shifts accuracy by about 3%. Check the per-class report in `inference_metrics.json` when comparing data versions.